# Scalable and Explainable Machine Learning for IoT Attack Detection

**Dataset:** CIC-IoT-2023 (Canadian Institute for Cybersecurity, University of New Brunswick)  
**Author:** Sumair Ahmed, Department of Artificial Intelligence, University of Sindh  
**Tools:** Python, pandas, scikit-learn, LightGBM, XGBoost (including its built-in TreeSHAP), run on a Kaggle CPU notebook

---

### What this notebook is
This is my end-to-end demo project. I chose the topic, found the dataset and built the pipeline myself. Every step is written out below together with the reasoning behind it, so the notebook can be read from top to bottom without any slides.

### Main results
- **8 attack categories:** macro-F1 of 0.841 for Random Forest, 0.832 for XGBoost and 0.789 for LightGBM. Accuracy is 0.994 to 0.995 for all three, which hides the differences.
- **Attack vs benign (2 classes):** macro-F1 of 0.959 to 0.970. **Exact attack type (34 classes):** 0.809 for Random Forest and 0.828 for XGBoost.
- **Weak spots:** rare attacks (Web, BruteForce and several single labels with fewer than 60 test rows).
- **Dependence on three features:** all models rely strongly on `IAT`, `Number` and `Weight`. Without them the 8-class macro-F1 falls to 0.66 to 0.69, so I report both scores.

### Roadmap
**Part 1: data and models**  
1 Research questions · 2 Dataset · 3 Setup · 4 Loading a sample · 5 Sanity checks · 6 Label distribution · 7 Grouping the labels · 8 Cleaning · 9 Exploratory data analysis · 10 Feature sets, split and baseline · 11 How the models are evaluated · 12 Random Forest · 13 LightGBM · 14 XGBoost · 15 Does the task difficulty matter? · 16 Saving the results and models

**Part 2: analysis**  
17 Reload check · 18 Results at a glance · 19 Explainability with SHAP · 20 Conclusion · 21 Limitations and future work · 22 Reproducibility and references

## 1. Research questions

IoT devices such as cameras, smart plugs and sensors are cheap, rarely patched and always online. That makes them easy targets for DDoS floods, port scans and botnets like Mirai. A useful intrusion detector for this traffic has to do two things: catch the rare attacks, and stay cheap enough to run close to the device.

I want to answer three questions:

- **RQ1.** Which tree-based model gives the best balance between detection quality (macro-F1) and cost (training and prediction time): Random Forest, LightGBM or XGBoost?
- **RQ2.** How does performance change as the task gets harder: attack vs benign (2 classes), attack category (8 classes), exact attack type (34 classes)?
- **RQ3.** Which flow features does the XGBoost model actually rely on, and does that make sense from a networking point of view? (answered with SHAP)

A comparison with a neural network was also planned, but it is not part of this notebook. It is listed under future work.

The main metric is **macro-F1**, not accuracy. The classes are extremely imbalanced, so accuracy would mostly measure how well the model predicts the biggest classes.

## 2. Dataset

CIC-IoT-2023 was built by the Canadian Institute for Cybersecurity at the University of New Brunswick. The authors ran 33 attacks against a lab of 105 real IoT devices (cameras, smart plugs, speakers, sensors, hubs, Raspberry Pis). The attacks are launched *from* compromised IoT devices rather than normal PCs, which makes the setting more realistic than older datasets.

| Property | Value |
|---|---|
| Devices | 105 |
| Attack types | 33, grouped into 7 categories |
| Categories | DDoS, DoS, Recon, Web-based, Brute-force, Spoofing, Mirai |
| Full size | about 46.6 million labelled flows (13.76 GB of CSV on Kaggle, 169 part files) |
| Features | flow-level statistics (packet sizes, TCP flag counts, rates, protocol indicators, inter-arrival time) plus a label |

**Where I got it:** the Kaggle mirror by Madhav Malhotra (`madhavmalhotra/unb-cic-iot-dataset`). The original is hosted by UNB CIC.

**Citation:** Neto, E. C. P., Dadkhah, S., Ferreira, R., Zohourian, A., Lu, R., Ghorbani, A. A. (2023). *CICIoT2023: A real-time dataset and benchmark for large-scale attacks in IoT environment.* Sensors, 23(13), 5941. https://doi.org/10.3390/s23135941

**Why I do not train on all 46 million rows:** a free Kaggle session has limited RAM and time, and many published papers also work on subsets of 1 to 2 million rows. I take a random 5% from every one of the 169 part files, which gives 2,334,322 rows and keeps all 34 labels represented. Training on the full data is listed as future work.

## 3. Setup

I import all libraries in one place (so a missing library fails at the start and not after a long run), fix a random seed so every run gives the same sample and the same split, and keep the settings, paths and plot style together.

**Settings at the top of the cell**
- `QUICK = True` runs a small dry run (every 6th csv file) to test the notebook in a few minutes. The real run uses `QUICK = False`.
- `RUN_34CLASS` switches the slowest part (the 34-class models) on or off.
- `RUN_SPLIT_CHECK` is an optional stability check on other random splits (section 15.4).

All figures are also saved as PNG files in `/kaggle/working/figures`, so I can reuse them in the README.

In [ ]:
import os
import glob
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
T_START = time.time()

# ---- settings ---------------------------------------------------------------
QUICK = False             # True: small dry run to test the notebook. False: the real run
RUN_34CLASS = True        # the 34-class models are the slowest part (about 10 minutes)
RUN_SPLIT_CHECK = False   # optional: repeat the 8-class models on 3 other random splits (about 15 minutes)

SEED = 42
SAMPLE_FRAC = 0.05        # share of every csv file that is kept
LABEL_COL = "label"
ROOT = "/kaggle/input/datasets/madhavmalhotra/unb-cic-iot-dataset/wataiData/csv/CICIoT2023"
WORK = "/kaggle/working"
FIG_DIR = f"{WORK}/figures"
SAMPLE_NAME = "ciciot_sample_quick.parquet" if QUICK else "ciciot_sample.parquet"
os.makedirs(FIG_DIR, exist_ok=True)
np.random.seed(SEED)

# ---- plot style -------------------------------------------------------------
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
CAT_ORDER = ["Benign", "DDoS", "DoS", "Mirai", "Recon", "Spoofing", "BruteForce", "Web"]
PALETTE = dict(zip(CAT_ORDER, sns.color_palette("colorblind", len(CAT_ORDER))))
SHORTCUTS = ["IAT", "Number", "Weight"]

### 3.1 Helper functions
Three small helpers are defined once and used everywhere: the rule that maps a label to its category, a function that shows a figure and also saves it as a PNG file, and a function that draws up to `n` random rows from every group (used for the SHAP sample and for the demo file).

In [ ]:
def to_category(label):
    # one label in, one category out
    if label == "BenignTraffic":
        return "Benign"
    if label.startswith("DDoS-"):
        return "DDoS"
    if label.startswith("DoS-"):
        return "DoS"
    if label.startswith("Mirai-"):
        return "Mirai"
    if label.startswith("Recon-") or label == "VulnerabilityScan":
        return "Recon"
    if label in ("MITM-ArpSpoofing", "DNS_Spoofing"):
        return "Spoofing"
    if label == "DictionaryBruteForce":
        return "BruteForce"
    if label in ("SqlInjection", "CommandInjection", "XSS",
                 "BrowserHijacking", "Backdoor_Malware", "Uploading_Attack"):
        return "Web"
    return "UNMAPPED"


def save_show(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def sample_per_group(df, col, n, seed=SEED):
    parts = [g.sample(min(len(g), n), random_state=seed) for _, g in df.groupby(col)]
    return pd.concat(parts)

## 4. Loading a sample of the data

The full dataset is about 46.6 million rows (13.76 GB of CSV), which is too much for a free notebook session. Instead I read every one of the 169 files and keep a random 5% of each, which gives about 2.3 million rows with all attack types present.

To save memory I convert `float64` columns to `float32`. The sample is saved once as a Parquet file, which is compressed and reloads in seconds instead of the roughly 5 minutes the CSV files need. If a saved sample already exists (in the working folder, or attached as an input from an earlier session), the cell loads it. Otherwise it builds the sample. The cell should not be interrupted while it builds the sample.

In [ ]:
def find_saved_sample():
    hits = glob.glob(f"{WORK}/{SAMPLE_NAME}") + glob.glob(f"/kaggle/input/**/{SAMPLE_NAME}", recursive=True)
    return hits[0] if hits else None


saved = find_saved_sample()
if saved:
    data = pd.read_parquet(saved)
    print("loaded the saved sample:", saved)
else:
    files = sorted(glob.glob(ROOT + "/part-*.csv"))
    assert files, "no csv files found: check ROOT and that the dataset is attached"
    if QUICK:
        files = files[::6]
    print(len(files), "csv files to read (about 5 minutes for all 169, please do not interrupt)")

    parts = []
    t0 = time.time()
    for i, f in enumerate(files):
        chunk = pd.read_csv(f).sample(frac=SAMPLE_FRAC, random_state=SEED)
        f64 = chunk.select_dtypes("float64").columns
        chunk[f64] = chunk[f64].astype("float32")
        parts.append(chunk)
        if i % 20 == 0:
            print(f"file {i:3d} done, {time.time() - t0:.0f}s elapsed")

    data = pd.concat(parts, ignore_index=True)
    del parts
    data.to_parquet(f"{WORK}/{SAMPLE_NAME}")

print("shape:", data.shape)
print(f"memory: {data.memory_usage(deep=True).sum() / 1e6:.0f} MB")
data.head()

## 5. Sanity checks

Before modelling I check four things that quietly break results if ignored:

- **Missing values (NaN).**
- **Infinite values.** Many flow features are rates, so a division by zero can produce `inf`.
- **Duplicate rows.** If the same row lands in both train and test, the score is inflated.
- **Column types.** Everything except the label should be numeric.

In [ ]:
numeric = data.select_dtypes("number")
checks = pd.Series({
    "rows": len(data),
    "columns": data.shape[1],
    "missing values": int(data.isna().sum().sum()),
    "infinite values": int(np.isinf(numeric).sum().sum()),
    "duplicate rows": int(data.duplicated().sum()),
    "memory (MB)": round(data.memory_usage(deep=True).sum() / 1e6),
})
print(checks.to_string())
print()
print(data.dtypes.value_counts())

### Result of the sanity checks

- The sample has 2,334,322 rows and 47 columns (46 numeric features stored as `float32`, plus the text label) and takes about 580 MB of memory.
- There are **no missing values and no infinite values**, so no imputation is needed.
- There are **370,543 duplicate rows (15.9%)**. They are removed in section 8, before the split.

## 6. Label distribution

This plot drives most of the later design choices. I use a log scale because the biggest class is thousands of times larger than the smallest, and on a normal axis the small classes would be invisible.

With an imbalance this extreme:
1. plain accuracy is misleading, so I report macro-F1 and per-class recall,
2. the split must be stratified so rare classes appear in both train and test,
3. the models need class weights (or resampling), otherwise they ignore the rare attacks.

In [ ]:
counts = data[LABEL_COL].value_counts()
print(len(counts), "distinct labels")
print("largest / smallest class: %.0f : 1" % (counts.max() / counts.min()))

order = counts.sort_values().index
colors = [PALETTE[to_category(l)] for l in order]

fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(order, counts[order].values, color=colors)
ax.set_xscale("log")
for y, v in enumerate(counts[order].values):
    ax.text(v * 1.06, y, f"{v:,}", va="center", fontsize=7.5)
ax.set_xlim(right=counts.max() * 4)
ax.set_xlabel("rows in the sample (log scale)")
ax.set_title("Class distribution: 34 labels, coloured by attack category")
handles = [plt.Rectangle((0, 0), 1, 1, color=PALETTE[c]) for c in CAT_ORDER]
ax.legend(handles, CAT_ORDER, title="category", loc="lower right")
save_show(fig, "01_label_distribution")

### Result: the classes are extremely imbalanced

There are 34 labels. The largest, `DDoS-ICMP_Flood`, has 360,901 rows and the smallest, `Uploading_Attack`, only 62, a ratio of 5,821 to 1. Most of the large labels are DDoS and DoS floods; the labels with fewer than 300 rows are web attacks (`BrowserHijacking`, `SqlInjection`, `CommandInjection`, `XSS`, `Backdoor_Malware`, `Uploading_Attack`) and `Recon-PingSweep`.

## 7. Grouping the 34 labels into 7 attack categories

The dataset has 34 labels: 33 attacks plus benign traffic. For the 8-class task I group them into the 7 categories used by the dataset authors, plus `Benign`.

| Category | What it contains |
|---|---|
| DDoS | distributed floods (SYN, UDP, TCP, ICMP, HTTP, SlowLoris and fragmentation variants) |
| DoS | the same style of flood from a single source |
| Mirai | botnet traffic (GREIP flood, GREETH flood, UDP plain) |
| Recon | scanning (host discovery, OS scan, port scan, ping sweep, vulnerability scan) |
| Spoofing | ARP spoofing (MITM) and DNS spoofing |
| BruteForce | dictionary brute force |
| Web | SQL injection, command injection, XSS, browser hijacking, backdoor, malicious upload |

The rule function `to_category` is defined with the helper functions in section 3, so that the label plot in section 6 could already use the category colours. Here I apply it and add an `assert` so that no label can end up unmapped without me noticing. This gives three targets from the same rows: `is_attack` (2 classes), `category` (8 classes) and `label` (34 classes).

In [ ]:
data["category"] = data[LABEL_COL].map(to_category)
data["is_attack"] = (data["category"] != "Benign").astype("int8")

unmapped = data.loc[data["category"] == "UNMAPPED", LABEL_COL].unique()
print("unmapped labels:", list(unmapped))
assert len(unmapped) == 0, "fix the mapping before going further"

cat_counts = data["category"].value_counts().reindex(CAT_ORDER)
table = pd.DataFrame({
    "labels": data.groupby("category")[LABEL_COL].nunique().reindex(CAT_ORDER),
    "rows": cat_counts,
    "share %": (cat_counts / len(data) * 100).round(2),
})
print()
print(table)
print()
print("attack share: %.2f%%" % (data["is_attack"].mean() * 100))

### Result: share of each category

No label was left unmapped. Before cleaning the categories have these shares: DDoS 72.8%, DoS 17.3%, Mirai 5.6%, Benign 2.4%, Spoofing 1.0%, Recon 0.8%, Web 0.05% and BruteForce 0.03%. In total 97.65% of the rows are attacks.

## 8. Cleaning

The sanity check showed no missing values and no infinite values, so no imputation is needed. The only real problem is duplicate rows (about 16%), which I handle in this order:

1. **Drop exact duplicate rows before splitting.** Otherwise the same row can sit in both train and test and the test score becomes optimistic.
2. **Drop constant columns**, because a column with one unique value carries no information.
3. **Check for conflicting rows**: rows with identical features but a different label. These cannot be separated by any model, so they would show where the ceiling of the dataset is.

I also compare the class sizes before and after, because removing duplicates hits flood attacks much harder than rare attacks, and the class balance changes.

In [ ]:
before = len(data)
before_counts = data["category"].value_counts()

data = data.drop_duplicates().reset_index(drop=True)
removed = before - len(data)
print(f"duplicates removed: {removed:,} ({removed / before:.1%})")
print(f"rows left: {len(data):,}")

target_cols = [LABEL_COL, "category", "is_attack"]
feature_cols = [c for c in data.columns if c not in target_cols]

constant = [c for c in feature_cols if data[c].nunique() <= 1]
print("constant columns:", constant)
feature_cols = [c for c in feature_cols if c not in constant]
data = data[feature_cols + target_cols]
print("features kept:", len(feature_cols))

# rows whose features match another row but whose label is different
conflicts = int(data.duplicated(subset=feature_cols, keep=False).sum())
print(f"rows sharing identical features with another row: {conflicts:,}")

# what did dropping duplicates do to each category
after_counts = data["category"].value_counts()
change = pd.DataFrame({"before": before_counts, "after": after_counts}).reindex(CAT_ORDER)
change["lost %"] = ((1 - change["after"] / change["before"]) * 100).round(1)
print()
print(change)
print()
print("attack share after cleaning: %.2f%%" % (data["is_attack"].mean() * 100))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(change.index, change["lost %"], color=[PALETTE[c] for c in change.index])
for i, v in enumerate(change["lost %"]):
    ax.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_ylabel("rows removed (%)")
ax.set_title("Share of rows removed as duplicates, per category")
save_show(fig, "02_duplicates_per_category")

### Result of the cleaning

- **370,543 exact duplicates removed (15.9%), 1,963,779 rows left.** Almost all of them are floods: DDoS lost 20.1% of its rows, DoS 5.4% and Mirai 5.0%. Benign, Spoofing, Web and BruteForce lost none, and Recon only 3 rows.
- **Two constant columns were removed** (`Telnet` and `SMTP`), so 44 features remain.
- **No conflicting rows:** no two rows have the same features but different labels.
- After cleaning, attacks are 97.20% of the data and benign traffic 2.8%.

## 9. Exploratory data analysis

### 9.1 Attack categories after cleaning
First look at how many rows each of the 8 categories has now. I use a log scale on the y-axis because DDoS has more than 1 million rows while BruteForce has fewer than 700.

In [ ]:
cat_counts = data["category"].value_counts().reindex(CAT_ORDER)
share = cat_counts / cat_counts.sum() * 100

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(cat_counts.index, cat_counts.values, color=[PALETTE[c] for c in cat_counts.index])
ax.set_yscale("log")
for i, (v, s) in enumerate(zip(cat_counts.values, share.values)):
    ax.text(i, v * 1.12, f"{v:,}\n{s:.2f}%", ha="center", va="bottom", fontsize=8)
ax.set_ylim(top=cat_counts.max() * 6)
ax.set_ylabel("rows (log scale)")
ax.set_title("Rows per category after cleaning")
save_show(fig, "03_category_counts")

### 9.2 Do attacks look different from each other?
If a feature has a clearly different average for different categories, a tree can split on it easily. I compare the **median** of a few features per category (the median is not distorted by the extreme values that flood traffic produces). The features live on very different scales (`Rate` is in the thousands, `syn_count` is below 1), so each feature is scaled from 0 (smallest category value) to 1 (largest category value). This makes the colours comparable; the absolute values are not shown.

In [ ]:
shown = ["Rate", "flow_duration", "Header_Length", "AVG", "Std",
         "syn_count", "ack_count", "rst_count", "Number", "Weight"]

med = data.groupby("category")[shown].median().reindex(CAT_ORDER)
span = (med.max() - med.min()).replace(0, 1)
scaled = (med - med.min()) / span

fig, ax = plt.subplots(figsize=(11, 4.8))
sns.heatmap(scaled, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            cbar_kws={"label": "scaled median (0 = lowest category, 1 = highest)"}, ax=ax)
ax.set_ylabel("")
ax.set_title("Median feature value per category (each feature scaled to 0-1)")
save_show(fig, "04_feature_medians_per_category")

### 9.3 Features that look suspiciously constant
While looking at the first rows I noticed that `Number` (9.5) and `Weight` (141.55) have the same value in the attack rows shown, and that `IAT` is large (about 83 million) in all five of them. The table below checks this over the whole data: for each of the three features it shows the most common value and the share of rows that have it, separately for attack and benign traffic.

If a feature is nearly constant for attacks but different for benign traffic, a model can use it as a shortcut. That could say more about how the flows were recorded than about the attacks. This check does not change anything yet. It only tells me what to watch for when I read the model results and the SHAP plots.

In [ ]:
rows = []
for col in SHORTCUTS:
    for flag, name in [(0, "benign"), (1, "attack")]:
        s = data.loc[data["is_attack"] == flag, col].round(2)
        top_val = s.value_counts().index[0]
        rows.append({
            "feature": col,
            "group": name,
            "most common value": top_val,
            "share of rows %": round((s == top_val).mean() * 100, 1),
            "unique values": s.nunique(),
        })
shortcut_table = pd.DataFrame(rows)
print(shortcut_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=shortcut_table, x="feature", y="share of rows %", hue="group",
            palette={"benign": "#4c9a6a", "attack": "#d9534f"}, ax=ax)
ax.set_ylabel("rows with the most common value (%)")
ax.set_title("How often does the most common value appear?")
ax.set_ylim(0, 105)
save_show(fig, "05_shortcut_features")

### Result

- **`Number` and `Weight` behave the same way.** `Number` is 9.5 in 97.7% of the attack rows but 5.5 in only 50.2% of the benign rows, and `Weight` shows the same pattern (141.55 against 38.5). The two are almost the same information (correlation 0.9996).
- **`IAT` is not constant.** Attack rows have 8,735 different `IAT` values, and the most common value (0.0) appears in only 0.7% of them. So the 83 million seen in the first rows is not the whole picture. I look at its distribution in the next step.

### 9.3b How is `IAT` spread inside each category?
The table above shows that `IAT` is not constant, but in the first rows its values were all large. Here I look at the distribution per category: the smallest value, the 5th percentile, the median, the 95th percentile and the largest value, and the share of rows whose `IAT` lies between 80 and 90 million. The data card describes `IAT` as the time difference to the previous packet but gives no unit, so I only describe the numbers and do not guess what they mean.

In [ ]:
# spread of IAT inside every category
iat = (data.groupby("category")["IAT"]
       .describe(percentiles=[0.05, 0.5, 0.95])[["min", "5%", "50%", "95%", "max"]]
       .reindex(CAT_ORDER))
print(iat.round(0))

# share of rows with IAT between 80 and 90 million, per category
band = data["IAT"].between(8e7, 9e7)
print()
print((band.groupby(data["category"]).mean().reindex(CAT_ORDER) * 100).round(1))

# the same on a log scale, for a random subset of rows
tmp = data[["category", "IAT"]].sample(min(300_000, len(data)), random_state=SEED).copy()
tmp["log10(IAT + 1)"] = np.log10(tmp["IAT"].clip(lower=0) + 1)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=tmp, x="category", y="log10(IAT + 1)", order=CAT_ORDER, hue="category",
            palette=PALETTE, showfliers=False, legend=False, ax=ax)
ax.set_xlabel("")
ax.set_title("IAT per category (log scale, 300k random rows)")
save_show(fig, "06_iat_per_category")

### 9.4 Correlation between features
Many flow features are computed from the same packets (average, minimum, maximum and total packet size, for example), so I expect groups of strongly correlated columns. This does not hurt tree models much, but it matters for explainability. When two features carry the same information, SHAP splits the credit between them and each one looks less important than it really is. I list the strongly correlated pairs here so I can mention it when I read the SHAP plots.

I use a 200k-row random subset, because the full matrix on 2 million rows is slow and a subset gives almost the same numbers.

In [ ]:
sub = data[feature_cols].sample(min(200_000, len(data)), random_state=SEED)
corr = sub.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.3, xticklabels=True, yticklabels=True,
            cbar_kws={"shrink": 0.7, "label": "correlation"}, ax=ax)
ax.tick_params(labelsize=7)
ax.set_title("Feature correlation (random subset of up to 200k rows)")
save_show(fig, "07_feature_correlation")

# strongly correlated pairs, upper triangle only so each pair appears once
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().reset_index()
pairs.columns = ["feature_a", "feature_b", "corr"]
strong = pairs[pairs["corr"].abs() > 0.95].sort_values("corr", key=abs, ascending=False)
print(f"{len(strong)} pairs with |corr| > 0.95")
strong.head(15)

### 9.5 What the EDA tells me

- **Duplicates are almost all floods.** 20% of DDoS rows were duplicates, against 0% for Benign, Spoofing, Web and BruteForce. Flood attacks repeat the same packet pattern many times.
- **Benign traffic is only 2.8% of the data.** A classifier that predicts "attack" for everything would already reach about 97% accuracy, so accuracy cannot be the main metric. I use macro-F1.
- **Three features might act as shortcuts.** `Number` equals 9.5 in 97.7% of attack rows but in only 50.2% of benign rows. `Weight` behaves the same way, and `IAT`, `Number` and `Weight` are correlated at 0.996 or more. Whether this reflects real traffic or the way the flows were recorded cannot be decided from the data alone, so I test how much the models depend on them.
- **Some features are copies.** `Srate` is perfectly correlated with `Rate` (1.0) and `LLC` with `IPv` (1.0). Only 11 feature pairs have a correlation above 0.95.

**Decisions.** I remove `Srate` and `LLC`, which leaves 42 features (the "full" set). Then I train every model on two feature sets: the full set, and the same set without `IAT`, `Number` and `Weight` (39 features). If the scores stay close, the models learned patterns that do not depend on these three features. If they drop a lot, the result depends on them.

## 10. Feature sets, train/test split and baseline

### 10.1 Split
I use an **80/20 stratified split** on the 34-class label, so every attack type, including the rarest, appears in both sets in the same proportion as in the data. The 2-class and 8-class targets come from the same rows, which means all three tasks are evaluated on exactly the same test set and the results are comparable.

The test set stays untouched until the final evaluation.

**A limitation I am aware of.** The rows are flow windows from continuous captures, so neighbouring rows can be very similar. A random split can therefore be somewhat optimistic. A stricter split (for example by capture file) is listed as future work.

In [ ]:
# Srate and LLC are copies of Rate and IPv
COPIES = ["Srate", "LLC"]
FEATURES_FULL = [c for c in feature_cols if c not in COPIES]
FEATURES_NOSHORT = [c for c in FEATURES_FULL if c not in SHORTCUTS]
FEATURE_SETS = {"full": FEATURES_FULL, "no shortcuts": FEATURES_NOSHORT}
print("full feature set:", len(FEATURES_FULL), "| without the three features:", len(FEATURES_NOSHORT))

# a label needs a few rows to be split into train and test (only matters for a QUICK run)
label_counts = data[LABEL_COL].value_counts()
too_small = label_counts[label_counts < 5].index
if len(too_small):
    print("labels dropped because they have fewer than 5 rows:", list(too_small))
    data = data[~data[LABEL_COL].isin(too_small)].reset_index(drop=True)

X = data[FEATURES_FULL]
y34 = data[LABEL_COL]
y8 = data["category"]
y2 = data["is_attack"]

(X_train, X_test,
 y34_train, y34_test,
 y8_train, y8_test,
 y2_train, y2_test) = train_test_split(
    X, y34, y8, y2,
    test_size=0.20, stratify=y34, random_state=SEED)

TARGETS = {
    "2-class": (y2_train, y2_test),
    "8-class": (y8_train, y8_test),
    "34-class": (y34_train, y34_test),
}

print("train:", X_train.shape, "| test:", X_test.shape)
print("labels in train:", y34_train.nunique(), "| labels in test:", y34_test.nunique())
print("attack share  train: %.3f | test: %.3f" % (y2_train.mean(), y2_test.mean()))
print()
print("rarest labels in the test set:")
print(y34_test.value_counts().tail(6))

### 10.2 A "do nothing" baseline
Before any real model I check what a classifier gets by always predicting the most common class. This is the floor every real model has to beat, and it shows why accuracy alone is misleading on this data.

In [ ]:
rows = []
for task in ["2-class", "8-class"]:
    y_train, y_test = TARGETS[task]
    dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    pred = dummy.predict(X_test)
    rows.append({
        "task": task,
        "accuracy": accuracy_score(y_test, pred),
        "macro-F1": f1_score(y_test, pred, average="macro"),
    })

pd.DataFrame(rows).round(4)

### Result of the split and the baseline

- **Train:** 1,571,023 rows. **Test:** 392,756 rows, 42 features (39 without the three suspicious ones). All 34 labels appear in both sets and the attack share is 0.972 in both.
- **The rarest labels in the test set are very small:** `Uploading_Attack` 12 rows, `Recon-PingSweep` 21, `Backdoor_Malware` 31 and `XSS` 37. Results for these labels are unreliable.
- **The "do nothing" baseline** reaches an accuracy of 0.9720 on the 2-class task but a macro-F1 of only 0.4929. On the 8-class task it reaches 0.6913 accuracy and 0.1022 macro-F1. Every model has to beat these macro-F1 values.

## 11. How the models are evaluated

Every model is scored on the same untouched test set, with the same function, so the numbers are comparable. For each run I record:

- **Accuracy**, only for completeness. On this data it is dominated by the DDoS class.
- **Macro-F1**: the F1 score is computed for each class separately and then averaged, so a rare class like BruteForce counts as much as DDoS. This is my main metric.
- **Per-class recall**: of all the real attacks of a type, how many did the model catch.
- **Training time** and **prediction time**: since the aim is a detector that is cheap enough to run near the device, speed is part of the result and not only an extra.

All results are collected in one list, so that the final comparison table can be built from it.

In [ ]:
RESULTS = []   # one row per model run, the final tables are built from this
PRED = {}      # predictions of every run: (model, feature set, task) -> predicted labels
MODELS = {}    # the fitted 8-class models that I keep for the demo app


def evaluate(name, model, feat_name, task):
    # fit on the training rows, score on the test rows, keep the numbers and the predictions
    feats = FEATURE_SETS[feat_name]
    y_train, y_test = TARGETS[task]

    t0 = time.time()
    model.fit(X_train[feats], y_train)
    train_s = time.time() - t0

    t0 = time.time()
    pred = model.predict(X_test[feats])
    pred_s = time.time() - t0

    row = {
        "model": name,
        "task": task,
        "features": feat_name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "train_s": train_s,
        "predict_s": pred_s,
        "us_per_row": pred_s / len(X_test) * 1e6,
    }
    RESULTS.append(row)
    PRED[(name, feat_name, task)] = pred
    if task == "8-class" and feat_name == "full" and name in ("Random Forest", "XGBoost"):
        MODELS[name] = model
    print(f"{name} | {task} | {feat_name}: macro-F1 {row['macro_f1']:.4f}, "
          f"accuracy {row['accuracy']:.4f}, train {train_s:.0f}s, predict {pred_s:.1f}s")


def recall_table(task, labels, combos):
    # recall of every label for a list of (model, feature set) runs
    y_test = TARGETS[task][1]
    out = {f"{m} ({f})": recall_score(y_test, PRED[(m, f, task)], labels=labels, average=None)
           for m, f in combos}
    return pd.DataFrame(out, index=labels)


def plot_confusion(ax, y_true, y_pred, labels, title):
    # each row is a true class, each cell the share of it that was predicted as the column class
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", cbar=False, vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, annot_kws={"size": 8}, ax=ax)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")


LABELS8 = [c for c in CAT_ORDER if c in set(y8_test)]

## 12. Baseline: Random Forest (8 categories)

Random Forest is my baseline. It is easy to explain, needs little tuning and is a standard reference in intrusion detection papers. I use 50 trees to keep the run time reasonable on a free Kaggle CPU.

Because of the class imbalance I set `class_weight="balanced_subsample"`. Each tree sees a bootstrap sample, and rare classes get a larger weight inside it, so the forest cannot ignore BruteForce or Web just because they have so few rows.

### 12.1 All 42 features
The first run uses the full feature set.

In [ ]:
RF_PARAMS = dict(n_estimators=50, n_jobs=-1, class_weight="balanced_subsample", random_state=SEED)

evaluate("Random Forest", RandomForestClassifier(**RF_PARAMS), "full", "8-class")

### 12.2 The shortcut test
Same model, same settings, but without `IAT`, `Number` and `Weight`. These three features looked suspicious in the EDA (section 9.3), so this run shows how much of the score depends on them.

- If the score stays close to the first run, the model does not depend on them.
- If it drops a lot, the first result partly depends on them. That would not tell me why: they could be real traffic properties or a side effect of how the flows were recorded.

In [ ]:
evaluate("Random Forest", RandomForestClassifier(**RF_PARAMS), "no shortcuts", "8-class")

pd.DataFrame(RESULTS).round(4)

### 12.3 Where does the model fail?
The overall score hides the weak spots, so I look at the recall of each of the 8 categories, and at a confusion matrix. In the matrix each row is the true class and each cell shows what fraction of that class was predicted as each category, so the diagonal is the recall.

In [ ]:
rf_recall = recall_table("8-class", LABELS8, [("Random Forest", "full"), ("Random Forest", "no shortcuts")])
rf_recall.insert(0, "test rows", y8_test.value_counts()[LABELS8].values)
print(rf_recall.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 6.2))
for ax, feat, title in zip(axes, ["full", "no shortcuts"], ["all 42 features", "without IAT, Number, Weight"]):
    plot_confusion(ax, y8_test, PRED[("Random Forest", feat, "8-class")], LABELS8,
                   f"Random Forest, {title}")
save_show(fig, "08_confusion_random_forest")

### 12.4 Reading the Random Forest results

- **Accuracy hides the weak classes.** With all features the accuracy is 0.9945 but the macro-F1 is 0.8414. DDoS alone is 271,522 of the 392,756 test rows, so accuracy mostly measures DDoS. The "predict the most common class" baseline scored a macro-F1 of 0.1022, so the forest learned a lot, but not evenly.
- **Weak spots.** Web (recall 0.33) and BruteForce (0.43) are the hardest, followed by Recon (0.79) and Spoofing (0.83). Web has only 241 test rows and BruteForce 137, so these numbers are noisy. A possible reason for Web is that flow statistics cannot see the packet payload, while SQL injection or XSS differ from normal HTTP mostly in the payload. I have not tested this.
- **The shortcut test did not turn out as I expected.** Without `IAT`, `Number` and `Weight`, macro-F1 falls from 0.8414 to 0.6596 and accuracy from 0.9945 to 0.8831. Benign recall barely moves (0.978 to 0.951), so these features are not mainly what separates attack from benign. The drop is in DoS (recall 0.9993 to 0.6831) and in the rare classes (Web 0.33 to 0.04, BruteForce 0.43 to 0.15). They help to tell attack types apart.
- **What this means.** Either these features describe genuine traffic properties (`IAT` is described as packet timing and `Number` as the number of packets in the flow), which would make using them legitimate, or they partly reflect how the flows were generated. This experiment cannot tell the two apart, so I report both scores and look at how the model uses these features with SHAP later (section 19).
- **Time.** Training took about 135 to 140 s without these three features, against about 80 to 90 s with them (timings vary a little between runs on a shared CPU). A possible reason is that the trees have to grow deeper when the features are less informative. Prediction times were measured on the whole test set with all CPU cores, so they are not the latency of a single IoT device.

## 13. Gradient boosting: LightGBM

Random Forest builds many independent trees. Gradient boosting builds trees one after another, and each new tree tries to fix the mistakes of the ones before it. I expected boosting to be the stronger option here, because it often beats a random forest on tabular data. Section 14.2 shows whether that held.

LightGBM grows its trees leaf by leaf and uses histogram binning, which makes it fast on 1.5 million rows. I use `class_weight="balanced"` for the same reason as before: rare classes get a larger weight, so the model cannot ignore them. Both feature sets are used, with the same `run_model` function, so the numbers can be compared directly with the Random Forest.

In [ ]:
LGB_PARAMS = dict(n_estimators=200, learning_rate=0.1, num_leaves=63, class_weight="balanced",
                  n_jobs=-1, random_state=SEED, verbose=-1)

evaluate("LightGBM", lgb.LGBMClassifier(**LGB_PARAMS), "full", "8-class")
evaluate("LightGBM", lgb.LGBMClassifier(**LGB_PARAMS), "no shortcuts", "8-class")

## 14. Gradient boosting: XGBoost

XGBoost is the other standard boosted-tree library. It is also boosting, with level-wise tree growth and its own regularisation, so it is a useful second opinion on the same data.

Two practical points. XGBoost needs integer labels, so a small wrapper class encodes the category names inside `fit` and decodes them again in `predict`; that way `run_model` works unchanged. XGBoost also has no `class_weight` option, so the same "balanced" weights are passed as per-row sample weights.

In [ ]:
class XGBLabelled:
    # xgboost wants integer labels: encode them in fit, decode them in predict
    def __init__(self, **params):
        self.params = params
        self.enc = LabelEncoder()

    def fit(self, X, y):
        y_int = self.enc.fit_transform(y)
        weights = compute_sample_weight("balanced", y_int)   # xgboost has no class_weight option
        self.model = xgb.XGBClassifier(**self.params)
        self.model.fit(X, y_int, sample_weight=weights)
        return self

    def predict(self, X):
        return self.enc.inverse_transform(self.model.predict(X))


XGB_PARAMS = dict(n_estimators=150, max_depth=8, learning_rate=0.1, tree_method="hist",
                  n_jobs=-1, random_state=SEED)

evaluate("XGBoost", XGBLabelled(**XGB_PARAMS), "full", "8-class")
evaluate("XGBoost", XGBLabelled(**XGB_PARAMS), "no shortcuts", "8-class")

pd.DataFrame(RESULTS).round(4)

### 14.1 All 8-class models side by side
Two views of the same runs. On the left, the macro-F1 of every model with and without the three suspicious features. On the right, macro-F1 against prediction time on a log scale, where the best models are in the upper left corner. Below them, the recall of every model for each category.

In [ ]:
res = pd.DataFrame(RESULTS)
r8 = res[res["task"] == "8-class"]
MODEL_COLORS = {"Random Forest": "#3b6ea5", "XGBoost": "#d9822b", "LightGBM": "#4c9a6a"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

sns.barplot(data=r8, x="model", y="macro_f1", hue="features", order=list(MODEL_COLORS),
            palette={"full": "#3b6ea5", "no shortcuts": "#d9a441"}, ax=axes[0])
for p in axes[0].patches:
    if p.get_height() > 0:
        axes[0].text(p.get_x() + p.get_width() / 2, p.get_height() + 0.01, f"{p.get_height():.3f}",
                     ha="center", fontsize=8)
axes[0].set_ylim(0.4, 1.08)
axes[0].set_yticks(np.arange(0.4, 1.01, 0.1))
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False)
axes[0].set_xlabel("")
axes[0].set_ylabel("macro-F1 (8 classes)")
axes[0].set_title("Macro-F1 with and without IAT, Number, Weight")

for _, r in r8.iterrows():
    marker = "o" if r["features"] == "full" else "s"
    axes[1].scatter(r["predict_s"], r["macro_f1"], s=110, color=MODEL_COLORS[r["model"]], marker=marker)
    if r["features"] == "full":
        axes[1].annotate(r["model"], (r["predict_s"], r["macro_f1"]),
                         textcoords="offset points", xytext=(9, 5), fontsize=9)
axes[1].scatter([], [], marker="o", color="gray", label="all features")
axes[1].scatter([], [], marker="s", color="gray", label="without IAT, Number, Weight")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)
axes[1].set_xscale("log")
axes[1].set_xlabel("prediction time for the whole test set (s, log scale)")
axes[1].set_ylabel("macro-F1 (8 classes)")
axes[1].set_title("Quality against speed")
save_show(fig, "09_model_comparison_8class")

combos = [(m, f) for m in MODEL_COLORS for f in ("full", "no shortcuts")]
rec = recall_table("8-class", LABELS8, combos)

fig, ax = plt.subplots(figsize=(11, 4.6))
sns.heatmap(rec.T, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar_kws={"label": "recall"}, ax=ax)
ax.set_title("Recall per category (8 classes)")
save_show(fig, "10_recall_per_category")

### 14.2 Reading the boosting results

| Model | Macro-F1, all features | Macro-F1, without the three features | Training time (s) | Prediction time, whole test set (s) |
|---|---|---|---|---|
| Random Forest | 0.8414 | 0.6596 | about 80 to 95 | about 1 |
| XGBoost | 0.8315 | 0.6682 | about 165 to 205 | about 5 to 6 |
| LightGBM | 0.7892 | 0.6892 | about 145 to 160 | about 19 to 27 |

The macro-F1 values were identical in every run. The times are from my runs on a shared Kaggle CPU and change a little from run to run, so I give ranges.

- **My expectation was wrong.** In section 13 I expected boosted trees to be the stronger option. On this data the Random Forest has the highest macro-F1 and is by far the fastest at prediction.
- **RF and XGBoost are practically tied.** The gap is 0.01. Web has only 241 and BruteForce 137 test rows, so a few rows can move macro-F1 by about that much. I did not repeat the runs with several seeds, so I do not call a winner. LightGBM is clearly lower (0.79), but I used a first configuration without tuning, so this says little about LightGBM in general.
- **Why LightGBM predicts slowly (a guess, not tested).** Multi-class boosting builds one tree per class in every round: 200 rounds x 8 classes = 1,600 trees, against 50 trees in the forest.
- **Accuracy cannot separate the models.** All three have 0.994 to 0.995, while macro-F1 ranges from 0.79 to 0.84.
- **The drop without `IAT`, `Number` and `Weight` appears in all three algorithms** (to between 0.66 and 0.69). So it comes from the data and not from one algorithm.
- **Limits.** One train/test split, one seed, no tuning. Differences of about 0.01 should not be over-interpreted.

## 15. Does the picture change when the task changes? (RQ2)

So far everything was the 8-category task. Now the same models are trained on the other two targets:

- **2 classes (attack or benign).** The easiest task. Besides macro-F1 I report two recalls: how many attacks were caught, and how much benign traffic was correctly left alone. In a real detector the second number is the false-alarm side, so it matters as much as the first.
- **34 classes (exact attack type).** The hardest task. Some labels have very few test rows (`Uploading_Attack` has 12, `Recon-PingSweep` 21), so the numbers for those classes are unreliable and I read them with care.

For the 2-class task I run all three models on both feature sets. For the 34-class task I run only Random Forest and XGBoost on all features, to save time: multi-class boosting builds one tree per class in every round, so LightGBM would need 200 x 34 = 6,800 trees. XGBoost uses 100 rounds here (150 before) for the same reason, so its 34-class result is not exactly comparable with the 8-class one.

In [ ]:
for feat in ["full", "no shortcuts"]:
    evaluate("Random Forest", RandomForestClassifier(**RF_PARAMS), feat, "2-class")
    evaluate("LightGBM", lgb.LGBMClassifier(**LGB_PARAMS), feat, "2-class")
    evaluate("XGBoost", XGBLabelled(**XGB_PARAMS), feat, "2-class")

# the two kinds of mistake, counted in flows of the test set
rows = []
for (name, feat, task), pred in PRED.items():
    if task != "2-class":
        continue
    rows.append({
        "model": name,
        "features": feat,
        "attack recall": recall_score(y2_test, pred, pos_label=1),
        "benign recall": recall_score(y2_test, pred, pos_label=0),
        "false alarms": int(((y2_test == 0) & (pred == 1)).sum()),
        "missed attacks": int(((y2_test == 1) & (pred == 0)).sum()),
    })
err2 = pd.DataFrame(rows)
print()
print(err2.round(4).to_string(index=False))

### Chart: the two kinds of mistake (2-class task)
A false alarm is a benign flow that was flagged as an attack. A missed attack is an attack that was not flagged. The two bars of each model show how it trades one against the other (all features).

In [ ]:
full2 = err2[err2["features"] == "full"].set_index("model")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, color in zip(axes, ["false alarms", "missed attacks"], ["#d9534f", "#3b6ea5"]):
    ax.bar(full2.index, full2[col], color=color)
    for i, v in enumerate(full2[col]):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
    ax.set_title(col.capitalize() + " (all features)")
    ax.set_ylabel("flows in the test set")
save_show(fig, "11_two_kinds_of_mistake")

In [ ]:
if RUN_34CLASS:
    evaluate("Random Forest", RandomForestClassifier(**RF_PARAMS), "full", "34-class")
    XGB34 = {**XGB_PARAMS, "n_estimators": 100}
    evaluate("XGBoost", XGBLabelled(**XGB34), "full", "34-class")
else:
    print("34-class runs are switched off (RUN_34CLASS = False)")

### 15.1 Reading the 2-class results

- **Attack vs benign is almost solved, but the two kinds of mistake are not equal.** With all features macro-F1 is 0.959 to 0.970 and attack recall is at least 0.995 for every model. The Random Forest has the best macro-F1 but the lowest benign recall (0.949): about 560 of the 10,987 benign test flows were flagged as attacks, against about 50 for XGBoost (benign recall 0.9955). On the other side the Random Forest missed fewer attacks (about 725 against about 1,830 for XGBoost). So the forest leans towards catching attacks at the price of more false alarms, and XGBoost does the opposite. Which one is better depends on what a real deployment cares about more.
- **The three features matter much less here.** Without `IAT`, `Number` and `Weight` the 2-class macro-F1 drops by about 0.03 for every model (in the 8-class task the drop was 0.10 to 0.18). So these features are not mainly what separates attack from benign. They help to tell attack types apart. My suspicion from the EDA was therefore only partly right.
- **Without them the Random Forest is the weakest on benign traffic** (benign recall 0.890, roughly 1,200 false alarms).

### Chart: recall of every label (34-class task)
Left: the recall of Random Forest and XGBoost for each of the 34 labels. The rarest labels are at the top and the number of test rows is in brackets. Right: recall against the number of test rows (log scale). Labels with few rows are the ones that are recognised worst, and their values are also the noisiest.

In [ ]:
if RUN_34CLASS:
    LABELS34 = sorted(y34_test.unique())
    n_test = y34_test.value_counts()
    rec34 = recall_table("34-class", LABELS34, [("Random Forest", "full"), ("XGBoost", "full")])
    rec34["test rows"] = n_test[LABELS34].values
    print(rec34.sort_values("test rows").round(3))

    order = list(rec34.sort_values("test rows", ascending=False).index)   # rarest labels end up at the top
    y = np.arange(len(order))
    rf_v = rec34.loc[order, "Random Forest (full)"].values
    xg_v = rec34.loc[order, "XGBoost (full)"].values

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9), gridspec_kw={"width_ratios": [1.5, 1]})
    for i in range(len(order)):
        ax1.plot([rf_v[i], xg_v[i]], [y[i], y[i]], color="lightgray", zorder=1)
    ax1.scatter(rf_v, y, color=MODEL_COLORS["Random Forest"], label="Random Forest", zorder=2)
    ax1.scatter(xg_v, y, color=MODEL_COLORS["XGBoost"], label="XGBoost", zorder=2)
    ax1.set_yticks(y)
    ax1.set_yticklabels([f"{l} ({n_test[l]:,})" for l in order], fontsize=8)
    ax1.set_xlim(-0.02, 1.03)
    ax1.set_xlabel("recall")
    ax1.set_title("Recall per label (test rows in brackets)")
    ax1.legend(loc="lower left")

    for col, name in [("Random Forest (full)", "Random Forest"), ("XGBoost (full)", "XGBoost")]:
        ax2.scatter(rec34["test rows"], rec34[col], color=MODEL_COLORS[name], label=name, alpha=0.85)
    ax2.set_xscale("log")
    ax2.set_ylim(-0.02, 1.03)
    ax2.set_xlabel("test rows of the label (log scale)")
    ax2.set_ylabel("recall")
    ax2.set_title("Rare labels are recognised worst")
    ax2.legend(loc="lower right")
    save_show(fig, "12_recall_per_label_34class")

### 15.2 Reading the 34-class results

- **Floods are easy, everything else is hard.** All DDoS, DoS and Mirai labels have a recall of 0.96 or higher with both models. The errors are in the small classes and in the Recon, Spoofing and Web labels.
- **XGBoost is much better on rare labels.** `Recon-PingSweep` 0.714 vs 0.095, `SqlInjection` 0.385 vs 0.019, `BrowserHijacking` 0.724 vs 0.172, `DictionaryBruteForce` 0.664 vs 0.182. These labels have only 21 to 137 test rows, so single numbers are noisy, but the direction is the same for most labels (`XSS` is an exception).
- **It has a price.** XGBoost recognises only 0.881 of benign flows correctly (Random Forest 0.990), so about 1,300 benign flows are flagged as some attack. A possible reason is that the balanced weights give rare labels a very large weight and pull predictions towards them. I have not tested this.
- Accuracy is 0.992 for both models, so accuracy cannot tell them apart. Macro-F1 can (0.809 vs 0.828).

In [ ]:
# macro-F1 for the three tasks
res = pd.DataFrame(RESULTS)
full = res[res["features"] == "full"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, color in MODEL_COLORS.items():
    d = full[full["model"] == name].set_index("task").reindex(["2-class", "8-class", "34-class"]).dropna(subset=["macro_f1"])
    ax.plot(d.index, d["macro_f1"], marker="o", color=color, label=name, linewidth=2)
    for x, v in zip(d.index, d["macro_f1"]):
        ax.text(x, v + 0.008, f"{v:.3f}", ha="center", fontsize=8, color=color)
ax.set_ylim(0.7, 1.0)
ax.set_ylabel("macro-F1")
ax.set_title("Macro-F1 gets lower as the task gets finer (all features)")
ax.legend()
save_show(fig, "13_macro_f1_by_task")

### 15.3 Answer to RQ2: what happens when the task gets harder?

| Task | Random Forest (macro-F1) | XGBoost (macro-F1) |
|---|---|---|
| 2 classes (attack / benign) | 0.970 | 0.959 |
| 8 classes (category) | 0.841 | 0.832 |
| 34 classes (exact attack) | 0.809 | 0.828 |

Performance falls as the task gets finer, and the biggest fall is from 2 to 8 classes. From 8 to 34 classes it falls only a little, probably because many of the extra labels are flood variants (SYN, UDP, TCP, ICMP) that are easy to recognise. The cost of the finer task sits in the rare labels.

**Limits.** One train/test split and one seed. In the 34-class run XGBoost used 100 rounds instead of 150, so the comparison with the 8-class run is not exact. Several labels have fewer than 60 test rows.

### 15.4 Optional: how stable are the results across splits?
All results so far come from one random split. To see how much the numbers move, this cell (switched off by default with `RUN_SPLIT_CHECK`) repeats the 8-class Random Forest and XGBoost on three other random splits and reports the mean and spread of the macro-F1. If the spread is larger than the gap between the two models, I should not call a winner.

In [ ]:
if RUN_SPLIT_CHECK:
    rows = []
    for split_seed in (1, 2, 3):
        Xa, Xb, ya, yb, ca, cb = train_test_split(X, y34, y8, test_size=0.20, stratify=y34, random_state=split_seed)
        for name, model in [("Random Forest", RandomForestClassifier(**RF_PARAMS)),
                            ("XGBoost", XGBLabelled(**XGB_PARAMS))]:
            model.fit(Xa, ca)
            rows.append({"model": name, "split seed": split_seed,
                         "macro_f1": f1_score(cb, model.predict(Xb), average="macro")})
        print("split", split_seed, "done")
    split_df = pd.DataFrame(rows)
    print(split_df.round(4).to_string(index=False))
    print()
    print(split_df.groupby("model")["macro_f1"].agg(["mean", "std", "min", "max"]).round(4))
else:
    print("split check is switched off (RUN_SPLIT_CHECK = False)")

## 16. Saving the results and the model files for the demo app

The two candidate models for the demo app (8 categories, all features) are the ones trained in sections 12 and 14. They are saved here: XGBoost (about 3 MB) and Random Forest (about 38 MB). Both are far below GitHub's limit of 100 MB per file. Also saved are:

- `results.csv`: every run of this notebook in one table,
- `model_meta.json`: the feature order and class names the app needs,
- `demo_sample.csv`: up to 100 test rows per category (800 rows), so that the app can be tried without the full dataset.

The cell also lists all saved files and counts the figures.

In [ ]:
# results of every run
res = pd.DataFrame(RESULTS)
res.to_csv(f"{WORK}/results.csv", index=False)

# the two candidate models for the demo app (kept from the 8-class runs, nothing is trained again)
for name, fname in [("XGBoost", "xgb_8class.joblib"), ("Random Forest", "rf_8class.joblib")]:
    path = f"{WORK}/{fname}"
    joblib.dump(MODELS[name], path, compress=3)
    print(f"{fname}: {os.path.getsize(path) / 1e6:.1f} MB")

# feature order and class names for the app
meta = {"features": FEATURES_FULL, "classes_8": sorted(y8_train.unique())}
with open(f"{WORK}/model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# small demo file: up to 100 test rows per category
demo = sample_per_group(X_test.join(y8_test.rename("true_category")), "true_category", 100)
demo.to_csv(f"{WORK}/demo_sample.csv", index=False)
print("demo sample:", demo.shape)

print()
print("files in /kaggle/working:")
for f in sorted(os.listdir(WORK)):
    p = os.path.join(WORK, f)
    if os.path.isfile(p):
        print(f"  {f}: {os.path.getsize(p) / 1e6:.1f} MB")
print("figures:", len(os.listdir(FIG_DIR)))

# Part 2: Analysis of the results

Part 1 (sections 1 to 16) prepared the data and trained and compared the models. Part 2 (sections 17 to 22) checks that the saved files reproduce the result, summarises all runs, explains the XGBoost model with SHAP and states the conclusions and limits.

## 17. Reload check

To be sure that a saved model file is a faithful copy, I load the saved XGBoost file back from disk and compare it with the model in memory: it must give exactly the same predictions and the same macro-F1 on the test set. The cell also prints the library versions, which I need for `requirements.txt`.

In [ ]:
reloaded = joblib.load(f"{WORK}/xgb_8class.joblib")
pred_reloaded = reloaded.predict(X_test[FEATURES_FULL])
pred_memory = PRED[("XGBoost", "full", "8-class")]

same = bool((pred_reloaded == pred_memory).all())
print("test set:", X_test.shape)
print("same predictions as the model in memory:", same)
print("macro-F1 of the reloaded model: %.4f" % f1_score(y8_test, pred_reloaded, average="macro"))
assert same, "the saved model does not match the model in memory"

for m in (pd, np, sklearn, xgb, lgb, joblib):
    print(m.__name__, m.__version__)

### Result of the reload check

The reloaded model gives exactly the same predictions as the model in memory (macro-F1 0.8315 on the 8-class task). The printed library versions are the ones listed in section 22 and in `requirements.txt`.

## 18. Results at a glance

All runs are in one table, and a chart shows what happens when the three suspicious features are removed (the chart of macro-F1 by task was shown in section 15). The tables below also give the speed of the 8-class models and the size of the drop for every model and task.

In [ ]:
res = pd.DataFrame(RESULTS)
res["task"] = pd.Categorical(res["task"], ["2-class", "8-class", "34-class"], ordered=True)

# macro-F1 for every model, feature set and task
print(res.pivot_table(index=["model", "features"], columns="task", values="macro_f1", observed=False).round(4))

# speed of the 8-class models (all features)
speed = res[(res["task"] == "8-class") & (res["features"] == "full")]
print()
print(speed[["model", "train_s", "predict_s", "us_per_row"]].round(1).to_string(index=False))

# effect of removing IAT, Number and Weight
both = res.pivot_table(index=["model", "task"], columns="features", values="macro_f1", observed=False).dropna()
both["drop"] = both["full"] - both["no shortcuts"]
print()
print(both.round(4))

g = sns.catplot(data=res[res["task"].isin(["2-class", "8-class"])], x="model", y="macro_f1",
                hue="features", col="task", kind="bar", height=4, aspect=1.1,
                palette={"full": "#3b6ea5", "no shortcuts": "#d9a441"})
g.set(ylim=(0.5, 1.0))
g.set_axis_labels("", "macro-F1")
g.fig.suptitle("Macro-F1 with and without IAT, Number, Weight", y=1.03)
g.savefig(f"{FIG_DIR}/14_effect_of_removing_three_features.png", dpi=150, bbox_inches="tight")
plt.show()

### Reading the charts

- **With all features** the Random Forest has the highest macro-F1 on the 2-class task (0.970) and the 8-class task (0.841). XGBoost is best on the 34-class task (0.828 against 0.809).
- **Removing `IAT`, `Number` and `Weight` costs every model.** About 0.03 on the 2-class task and 0.10 to 0.18 on the 8-class task. LightGBM loses the least and is slightly ahead without them (0.689 against 0.668 for XGBoost and 0.660 for Random Forest), but these differences are small.
- **Speed.** The Random Forest predicts the whole test set in about 1 second, XGBoost in about 5 to 6 seconds and LightGBM in about 20 seconds.

## 19. Explainability: what does the XGBoost model rely on? (RQ3)

SHAP gives every feature a contribution for every single prediction: how much did this feature push the model towards or away from a class. Averaging the absolute values over many rows gives a global importance ranking.

I use XGBoost's built-in SHAP computation for trees (`pred_contribs`) on a sample of up to 300 test rows per category (2,178 rows in total), so the rare categories (Web, BruteForce) are included and not drowned by DDoS.

I show three views: the global importance of every feature, a heatmap of the importance per category, and the top features of each category.

What I look for:
- Which features matter most overall?
- Do `IAT`, `Number` and `Weight` (the three suspicious features) carry a large share? If yes, the result depends on them.
- Are the important features different for each attack category, and does that make sense in networking terms?

One thing to remember when reading the plots: when two features carry the same information, SHAP splits the credit between them (for example `IAT`, `Number` and `Weight` are correlated at 0.996 or more, and `Std` and `Radius` at 0.99996). Each of them can look less important than the information they share really is.

In [ ]:
xgb_model = MODELS["XGBoost"]

# up to 300 test rows per category, so the rare categories are included
sample = sample_per_group(X_test.join(y8_test.rename("cat")), "cat", 300)
cats = sample.pop("cat").values
print("rows used:", len(sample))

# xgboost's own SHAP values, shape: rows x classes x (features + bias)
booster = xgb_model.model.get_booster()
contrib = booster.predict(xgb.DMatrix(sample), pred_contribs=True)
sv = contrib[:, :, :-1]                      # drop the bias column
classes = list(xgb_model.enc.classes_)
print("shap array:", sv.shape)

# global importance: mean |shap| over rows and classes
imp = pd.Series(np.abs(sv).mean(axis=(0, 1)), index=sample.columns).sort_values(ascending=False)
share = imp / imp.sum() * 100
print()
print("top 12 features, share of total importance (%):")
print(share.head(12).round(1))
print()
print("share of IAT + Number + Weight: %.1f%%" % share[SHORTCUTS].sum())

fig, ax = plt.subplots(figsize=(8, 6.5))
top = share.head(15)[::-1]
ax.barh(top.index, top.values, color=["#d9534f" if f in SHORTCUTS else "#3b6ea5" for f in top.index])
for y, v in enumerate(top.values):
    ax.text(v + 0.3, y, f"{v:.1f}%", va="center", fontsize=8)
ax.set_xlabel("share of total mean |SHAP| (%)")
ax.set_title("Global feature importance (red = IAT, Number, Weight)")
save_show(fig, "15_shap_global_importance")

# importance per category: which features push the model towards that category
per_class = {}
for k, c in enumerate(classes):
    rows = cats == c
    per_class[c] = np.abs(sv[rows, k, :]).mean(axis=0)
pc = pd.DataFrame(per_class, index=sample.columns)
pc = pc / pc.sum() * 100                      # each column sums to 100
pc = pc[[c for c in CAT_ORDER if c in pc.columns]]

print()
for c in pc.columns:
    print(c, "->", ", ".join(pc[c].sort_values(ascending=False).head(3).index))

fig, ax = plt.subplots(figsize=(9.5, 6.5))
sns.heatmap(pc.loc[imp.head(12).index], annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={"label": "% of that category's importance"}, ax=ax)
ax.set_title("Which features matter for which category")
save_show(fig, "16_shap_importance_per_category")

# the top 6 features of every category
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, c in zip(axes.ravel(), pc.columns):
    t = pc[c].sort_values(ascending=False).head(6)[::-1]
    ax.barh(t.index, t.values, color=["#d9534f" if f in SHORTCUTS else PALETTE[c] for f in t.index])
    ax.set_title(c)
    ax.tick_params(labelsize=8)
fig.suptitle("Top 6 features per category (red = IAT, Number, Weight)", y=1.01, fontweight="bold")
save_show(fig, "17_shap_top_features_per_category")

### 19.1 Reading the SHAP results

- **One feature dominates.** `IAT` alone makes up 28.8% of the model's total importance and `Number` 12.6%. Together with `Weight` (about 1.8%) the three suspicious features make up 43.2%.
- **`IAT` is in the top three for all 8 categories** (first for 7 of them, third for Mirai).
- **This fits the earlier test.** Without these three features the XGBoost 8-class macro-F1 falls from 0.83 to 0.67, so the model really depends on them.
- **Some other results look plausible in networking terms, but I did not verify them.** `fin_count` and `syn_count` are among the top three features for Recon, which fits scans being recognised by their TCP flag patterns. `SSH` is third for BruteForce, which fits dictionary attacks targeting remote login services. `Protocol Type` is the top feature for Mirai; I did not investigate why.
- **`IAT` is doubtful.** The data card describes it as the time difference to the previous packet but gives no unit. In the five attack rows shown at the top of the data, its values are between 83.0 and 83.4 million, which does not look like the gap between two packets of a fast flood. I do not know how the dataset's feature extractor computed it, so I cannot say whether it reflects real packet timing or a property of how the flows were built.
- **What I can claim.** The headline score (macro-F1 0.83) depends strongly on a feature whose meaning I cannot fully verify. The score without the three features (0.67) is a more cautious estimate of how well the other features separate the attacks. The real value probably lies between the two, so I report both.
- **Limits of this analysis.** SHAP shows what the model uses, not what causes an attack. The sample has up to 300 rows per category, so rare categories weigh more than in real traffic. Correlated features share credit, so single values should not be over-read. I did not test removing only `IAT`, which would show how much of the effect comes from it alone. That is future work.

## 20. Conclusion

I built a complete pipeline for attack detection on the CIC-IoT-2023 dataset, from a 5% sample of every file (2.33 million rows, 1.96 million after removing duplicates) to explained models.

**Answers to the research questions**
- **RQ1 (which tree model?).** Random Forest gave the best 8-class macro-F1 (0.841) and by far the fastest prediction (about 1 s for 392,756 flows). XGBoost was practically tied (0.832), was better on rare attack types in the 34-class task (0.828 against 0.809) and raised fewer false alarms on the 2-class task. LightGBM was lowest in my first configuration (0.789) and slowest at prediction. There is no single winner: it depends on whether catching every attack or avoiding false alarms matters more.
- **RQ2 (harder task).** Macro-F1 falls from about 0.96 to 0.97 (2 classes) to about 0.83 to 0.84 (8 classes) and to 0.81 to 0.83 (34 classes). The loss sits in the rare labels, not in the floods.
- **RQ3 (what do the models use?).** The XGBoost model relies mostly on `IAT` (28.8%) and `Number` (12.6%). I could not verify what `IAT` measures, so I report two scores: 0.83 with all features and 0.67 without `IAT`, `Number` and `Weight`.

**Practical points**
- Accuracy is above 0.99 for every model and says almost nothing here. Macro-F1 and per-class recall show where the models fail.
- The saved XGBoost model is only 3.0 MB and the Random Forest 38.2 MB, so both are small enough for a demo application.

## 21. Limitations and future work

**Limitations**
- One random 80/20 split and one seed, so differences of about 0.01 are not reliable. Neighbouring rows of a capture are similar, so a random split is probably somewhat optimistic.
- Only a 5% sample of the data, and no hyperparameter tuning (first configurations only).
- The rare labels have very few test rows (12 to 60), so their recall values are noisy.
- It is unclear what `IAT`, `Number` and `Weight` represent, and the models depend strongly on them.
- Prediction times were measured on the whole test set with all CPU cores of a shared machine. They are not the latency of one flow on an IoT device.
- SHAP was computed on a sample of 2,178 rows and only for the XGBoost model.
- No neural network was trained, so the comparison covers tree models only.

**Future work**
1. A stricter split (for example by capture file) to check how optimistic the random split is.
2. Remove `IAT` alone, and each of the three features separately, to see which one the result depends on, and check the dataset's documentation on how they were computed.
3. Repeat the runs with several seeds and report mean and spread. Tune the hyperparameters, especially for LightGBM.
4. Train a small neural network (MLP) and compare it with the tree models.
5. Use more of the 46.6 million rows.
6. Measure single-flow latency and memory on a small device such as a Raspberry Pi, with a compressed model.

## 22. Reproducibility and references

**Reproducibility**
- Random seed 42 for the sampling, the split and all models.
- Library versions: pandas 2.3.3, numpy 2.1.3, scikit-learn 1.6.1, xgboost 3.4.1, joblib 1.6.0 (LightGBM, matplotlib and seaborn as installed on the Kaggle image).
- Saved files: `ciciot_sample.parquet`, `results.csv`, `model_meta.json`, `demo_sample.csv`, `xgb_8class.joblib`, `rf_8class.joblib`. The raw dataset is not included; it can be downloaded from the sources in section 2.

**References**
- Neto, E. C. P., Dadkhah, S., Ferreira, R., Zohourian, A., Lu, R., Ghorbani, A. A. (2023). CICIoT2023: A real-time dataset and benchmark for large-scale attacks in IoT environment. *Sensors*, 23(13), 5941.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.
- Chen, T., Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of KDD 2016*, 785-794.
- Ke, G. et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems 30*.
- Lundberg, S. M., Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems 30*.

In [ ]:
print("whole notebook ran in %.1f minutes" % ((time.time() - T_START) / 60))